In [2]:
import pandas as pd
from graphviz import Digraph

# Read your Excel worksheet
df = pd.read_excel(
    "c:/Users/hermawan/OneDrive - Stichting Deltares/PhD/Egypt_ERF_data/egypt-survey-ml/notebooks/corelation_final.xls",
    sheet_name="summary",
)

# Convert correlation values to numbers and remove incomplete rows
df["femalecorel"] = pd.to_numeric(df["malecorel"], errors="coerce")
df = df.dropna(subset=["var1", "var2", "femalecorel"])

# Create the diagram
cld = Digraph("CLD", format="png")
cld.attr(
    layout="dot",
    rankdir="TB",
    size="11.69,8.27!",
    ratio="fill",
    dpi="150",
    nodesep="0.2",
    ranksep="0.3",
)
cld.attr("node", shape="box", style="rounded", fontsize="18", fontname="Arial")
cld.attr("edge", fontsize="16", fontname="Arial")
for _, row in df.iterrows():
    correlation = row["femalecorel"]

    if correlation == 0:
        continue  # Skip zero correlations

    polarity = "+" if correlation > 0 else "−"

    cld.edge(
        str(row["var1"]).strip(),
        str(row["var2"]).strip(),
        label=polarity,
    )

from IPython.display import HTML, display

svg = cld.pipe(format="svg").decode("utf-8")
display(HTML(
    f'<div style="overflow:auto; max-height:850px;">'
    f'<div style="width:max-content;">{svg}</div></div>'
))


output_path = "D:/male"

cld.render(
    filename=output_path,
    format="png",
    cleanup=True
)

print(f"Saved as: {output_path}.png")

Saved as: D:/male.png


In [19]:
import pandas as pd
import textwrap
from graphviz import Digraph
from IPython.display import HTML, display

df = pd.read_excel(
    "c:/Users/hermawan/OneDrive - Stichting Deltares/PhD/"
    "Egypt_ERF_data/egypt-survey-ml/notebooks/corelation_final.xls",
    sheet_name="summary",
)

# Settings: increase MIN_CORRELATION or decrease MAX_LINKS for fewer links
MIN_CORRELATION = 0.30
MAX_LINKS = 3

df["femalecorel"] = pd.to_numeric(df["femalecorel"], errors="coerce")
df = df.dropna(subset=["var1", "var2", "femalecorel"]).copy()

for col in ["var1", "var2"]:
    df[col] = df[col].astype(str).str.strip()

# Remove self-links, duplicates, and weak connections
df["strength"] = df["femalecorel"].abs()
df = (
    df[
        (df["var1"] != df["var2"])
        & df["var1"].ne("")
        & df["var2"].ne("")
        & (df["strength"] >= MIN_CORRELATION)
        & (df["strength"] > 0)
    ]
    .sort_values("strength", ascending=False)
    .drop_duplicates(["var1", "var2"])
    .groupby("var1", sort=False)
    .head(MAX_LINKS)
)

cld = Digraph("CLD", engine="neato", format="svg")

cld.attr(
    overlap="false",
    splines="true",
    sep="+12",
    start="42",
    maxiter="2000",
    outputorder="edgesfirst",
    bgcolor="white",
    pad="0.3",
)

cld.attr(
    "node",
    shape="box",
    style="rounded,filled",
    fillcolor="#F1F5F9",
    color="#CBD5E1",
    penwidth="1",
    fontname="Arial",
    fontsize="12",
    fontcolor="#1E293B",
    margin="0.14,0.09",
)

cld.attr(
    "edge",
    fontname="Arial",
    fontsize="11",
    arrowsize="0.6",
    penwidth="1.1",
    len="1.3",
)

for variable in pd.unique(df[["var1", "var2"]].values.ravel()):
    cld.node(
        variable,
        label=textwrap.fill(variable.replace("_", " "), width=22),
    )

for _, row in df.iterrows():
    positive = row["femalecorel"] > 0
    colour = "#3B82F6" if positive else "#E16B65"

    cld.edge(
        row["var1"],
        row["var2"],
        xlabel="+" if positive else "−",
        color=colour,
        fontcolor=colour,
    )

svg = cld.pipe().decode("utf-8")
display(HTML(
    f'<div style="overflow:auto; max-height:850px;">{svg}</div>'
))